# Customer Value Segmentation and 90-Day Repeat-Purchase Propensity

## Raw data ingestion section

This section loads the original UCI Online Retail II Excel workbook,
verifies its worksheets and schema, combines the two transaction periods,
and creates a schema-normalised Parquet file.

No transaction rows are cleaned or removed at this stage.

In [1]:
!pip install -q openpyxl pyarrow

In [2]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import requests

REPO_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

RAW_DATA_PATH = (
    REPO_ROOT
    / "data"
    / "interim"
    / "online_retail_raw.parquet"
)

RAW_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


UCI_DATA_URL = (
    "https://archive.ics.uci.edu/static/public/502/"
    "online%2Bretail%2Bii.zip"
)

response = requests.get(UCI_DATA_URL, timeout=120)
response.raise_for_status()

with ZipFile(BytesIO(response.content)) as z:
    with z.open("online_retail_II.xlsx") as f:
        raw_sheets = pd.read_excel(
            f,
            sheet_name=None,
            engine="openpyxl"
        )

print(list(raw_sheets))

['Year 2009-2010', 'Year 2010-2011']


In [3]:
transactions = pd.concat(
    [
        df.assign(source_sheet=sheet_name)
        for sheet_name, df in raw_sheets.items()
    ],
    ignore_index=True
)

print(transactions.shape)
transactions.head()

(1067371, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


In [4]:
EXPECTED_COLUMNS = {
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
}

In [5]:
missing = EXPECTED_COLUMNS - set(transactions.columns)

if missing:
    raise ValueError(f"Missing expected columns: {missing}")

In [6]:
transactions = transactions.rename(columns={
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "price",
    "Customer ID": "customer_id",
    "Country": "country"
})

In [7]:
string_columns = [
    "invoice",
    "stock_code",
    "description",
    "country",
    "source_sheet"
]

transactions[string_columns] = transactions[string_columns].astype("string")

In [8]:
transactions.dtypes

,0
invoice,string[python]
stock_code,string[python]
description,string[python]
quantity,int64
invoice_date,datetime64[ns]
price,float64
customer_id,float64
country,string[python]
source_sheet,string[python]


In [9]:
transactions.to_parquet(
    RAW_DATA_PATH,
    index=False
)

print(f"Saved to: {RAW_DATA_PATH}")

Saved to: /content/data/interim/online_retail_raw.parquet


In [10]:
print("Exists:", RAW_DATA_PATH.exists())
print(
    f"Size: "
    f"{RAW_DATA_PATH.stat().st_size / 1_000_000:.2f} MB"
)

Exists: True
Size: 7.29 MB


In [11]:
test_df = pd.read_parquet(
    RAW_DATA_PATH
)

print("Shape:", test_df.shape)
print()
print(test_df.dtypes)

Shape: (1067371, 9)

invoice         string[python]
stock_code      string[python]
description     string[python]
quantity                 int64
invoice_date    datetime64[ns]
price                  float64
customer_id            float64
country         string[python]
source_sheet    string[python]
dtype: object


In [12]:
assert len(test_df) == len(transactions)
assert list(test_df.columns) == list(transactions.columns)

print("Parquet round-trip validation passed.")

Parquet round-trip validation passed.
